# envs

In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch
import numpy as np
from net import *
import warnings
from collections import Counter
import pickle
from net import *
from utils import *

from imblearn import over_sampling, under_sampling,combine
from compare_model import *


# Binary

## func

In [9]:
def GetAllRes(proj, proj1, proj2):
    class GetArgs():
        def __init__(self):
            self.DataPath = '../data/'
            self.OutPath = '../result/'
            self.item = 'b'
            self.proj = proj
            self.proj1 = proj1
            self.proj2 = proj2 
            self.splRat = 0.8
            self.repeat = 1
    args = GetArgs()

    OutPath = args.OutPath
    proj = args.proj
    proj1 = args.proj1
    proj2 = args.proj2

    model_pre = OutPath + proj1 + '_vs_' + proj2
    x1, x1_test, y1, y1_test = load_data(args.DataPath, proj, args.proj1, args.proj2, args.splRat, args.repeat)
    pkl_fils = os.path.join(model_pre, f'model.pkl')

    with open(pkl_fils, 'rb') as pickle_file:
        loaded_data = pickle.load(pickle_file)
    model_1 = loaded_data[0]
    optimizer_1 = loaded_data[1]
    vnet_1 = loaded_data[2]
    optimizer_vnet_1 = loaded_data[3]

    # VnetPanting(vnet_1)

    results = []
    with torch.no_grad():
        yhat = model_1(x1_test).squeeze(-1).detach().numpy()
        # y_test_pred = norY(yhat)
        y_test_pred = norY(model_1(x1_test).squeeze(-1).detach().numpy()) 
        auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = print_eva(y1_test, y_test_pred, model_1(x1_test).squeeze(-1).detach().numpy(), 'test')
        results.append(['MWENA', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'elasticnet')
    results.append(['Original-Elasticnet', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])
    
    x_resampled, y_resampled = over_sampling.SMOTE().fit_resample(x1, y1)
    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x_resampled, y_resampled, x1_test, y1_test, model_meth = 'elasticnet')
    results.append(['Smote-Elasticnet', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'elasticnetCS')
    results.append(['Elasticnet-CS', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'RF')
    results.append(['RF', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'SVM')
    results.append(['SVM', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'KNN')
    results.append(['KNN', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'XGBoost')
    results.append(['XGBoost', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'AdaBoost')
    results.append(['AdaBoost', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = Compare_main_pip_cv(x1, y1, x1_test, y1_test, model_meth = 'LightGBM')
    results.append(['LightGBM', auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy])

    columns = ['Method', 'AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'Gmean', 'F1 Score', 'AUPRC', 'MCC', 'Balanced Accuracy']
    df = pd.DataFrame(results, columns=columns)

    return df

## result

In [ ]:
df_exoRBase = GetAllRes(proj = "exoRBase", proj1 = "Benign", proj2 = "CRC")

In [ ]:
df_exoRBase